# Reading -> Transforming -> Writing (from : bronze.erp_cust_az12_raw to silver.erp_cust_az12)

## Reading the date

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, trim, when, lit
from pyspark.sql.types import StringType

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12_raw")
df.printSchema()

## Data transformation

### Trimming empty spaces

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Renaming column

In [0]:
df = df.withColumnRenamed("GEN", "GENDER")

### Finding rows having incorrect or Null Gender

In [0]:
bad_gender = df.filter((col("GENDER").isNull()) | (~col("GENDER").isin("Male", "Female")))
bad_gender.groupBy("GENDER").count().display()

### Transforming M to Male and F to Female and empty to null

In [0]:

df = df.withColumn( "GENDER",
        when(col("GENDER") == "M", "Male")\
        .when(col("GENDER") == "F", "Female")\
        .when(col("GENDER") == '', lit(None))\
        .otherwise(col("GENDER"))
    ) 


### Validating date (null, incorrect, incorrect format, incorrect value)

In [0]:
string_bdate = col("BDATE").cast("string")
month = string_bdate.substr(6, 2).cast("int")
day = string_bdate.substr(9, 2).cast("int")
bad_date = df.filter((col("BDATE").isNull()) | ((month < 1) | (month > 12)) | (day < 1) | (day > 31))

bad_date.groupBy("BDATE").count().display()


### Checking if there is duplicate row based on CID

In [0]:
print(f"Total row count : {df.count()}")
duplicates = df.groupBy("CID").count().filter(col("count") > 1)
print(f"Duplicate count: {duplicates.count()}")

## Writing data to silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_cust_az12")